# RLHF: Aligning GPT-2 for Hedged Financial Language
# **PPO Implemented From Scratch**

## Goal
Train GPT-2 using RLHF to prefer **factual, hedged financial language** over **confident/speculative language** — with **PPO written entirely from scratch** (no `trl.PPOTrainer`).

**Example of what we want:**
- ✅ *"The stock may experience volatility given current market conditions"*
- ❌ *"The stock will definitely surge and reach new highs"*

## Pipeline Overview
```
Stage 1: SFT  →  Fine-tune GPT-2 on financial text prompts
Stage 2: RM   →  Train Reward Model on synthetic pairwise preferences  
Stage 3: PPO  →  RL fine-tuning — PPO implemented from scratch (no trl)
```

## PPO From-Scratch Architecture
```
PolicyWithValueHead     — GPT-2 backbone + scalar value head V(s)
RolloutBuffer           — stores (query, response, log_prob, value, reward, advantage)
compute_gae()           — Generalized Advantage Estimation (GAE-λ)
compute_kl_penalty()    — per-token KL divergence from reference (SFT) model
ppo_loss()              — clipped surrogate + value loss + entropy bonus
PPOTrainerScratch       — orchestrates rollout → score → update loop
```

## Setup & Installations

In [ ]:
# Install required libraries
# NOTE: trl is NOT used for PPO — we implement it from scratch
# trl is kept only because Stage 1 SFT uses HuggingFace Trainer (not trl)
!pip install transformers datasets accelerate torch -q

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional

from transformers import (
    GPT2LMHeadModel,
    GPT2Model,
    GPT2Tokenizer,
    GPT2ForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from datasets import Dataset

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

---
## Stage 0: Define the Preference Signal

Before any training, we define **what we mean by hedged financial language**.
This is the human preference we're trying to encode into a reward model.

### Hedging vocabulary
We use two lexicons:
- **Hedging words**: epistemic uncertainty markers common in careful financial writing
- **Speculative words**: overconfident, promotional, or predictive language to penalize

In [ ]:
# ── Preference Lexicons ────────────────────────────────────────────────────────

HEDGE_WORDS = [
    # Epistemic hedges
    "may", "might", "could", "possibly", "potentially",
    "appears to", "seems to", "tends to", "suggests",
    # Uncertainty markers
    "uncertain", "unclear", "subject to", "risk", "risks",
    "volatility", "fluctuation", "variability",
    # Conditionality
    "if", "depending on", "assuming", "provided that",
    "contingent", "subject to change",
    # Analytical qualifiers  
    "historically", "based on", "according to", "indicates",
    "analysis suggests", "data shows", "evidence suggests"
]

SPECULATIVE_WORDS = [
    # Overconfident predictions
    "will definitely", "will certainly", "guaranteed", "guarantee",
    "without a doubt", "absolutely will", "100%", "sure to",
    # Hype language  
    "skyrocket", "explode", "moon", "to the moon", "unstoppable",
    "massive gains", "huge profits", "get rich", "can't lose",
    # Imperative speculation
    "you must buy", "buy now", "sell immediately", "don't miss",
    "once in a lifetime", "never been a better time",
    # Sensationalism
    "amazing", "incredible returns", "unbelievable", "shocking"
]

def compute_hedge_score(text: str) -> float:
    """
    Heuristic reward: measures the hedging quality of financial text.
    
    Returns a score in [0, 1] where:
      1.0 = well-hedged, cautious financial language
      0.0 = speculative, overconfident language
    
    This is our proxy for human preference — in a real RLHF setup,
    human annotators would provide pairwise labels instead.
    """
    text_lower = text.lower()
    words = text_lower.split()
    
    if len(words) == 0:
        return 0.5  # neutral for empty text
    
    # Count hedge signal
    hedge_count = sum(
        1 for h in HEDGE_WORDS if h in text_lower
    )
    
    # Count speculative signal  
    spec_count = sum(
        1 for s in SPECULATIVE_WORDS if s in text_lower
    )
    
    # Penalize ALL-CAPS (hype marker)
    caps_ratio = sum(1 for c in text if c.isupper()) / max(len(text), 1)
    caps_penalty = caps_ratio * 2
    
    # Penalize excessive exclamation marks
    exclaim_penalty = min(text.count('!') * 0.3, 1.0)
    
    # Raw score: hedge signal minus speculative signal
    raw = hedge_count - (spec_count * 1.5) - caps_penalty - exclaim_penalty
    
    # Sigmoid normalization to [0, 1]
    score = 1 / (1 + np.exp(-raw))
    
    return float(score)


# ── Sanity check the scoring function ────────────────────────────────────────
test_cases = [
    ("The stock may experience volatility depending on interest rate decisions.", "GOOD"),
    ("Based on historical data, earnings could potentially miss estimates.", "GOOD"),
    ("Analysis suggests the sector might face headwinds if inflation persists.", "GOOD"),
    ("This stock will DEFINITELY skyrocket! Guaranteed massive gains!", "BAD"),
    ("BUY NOW before it's too late! You can't lose! To the moon!!!", "BAD"),
    ("Get rich quick with this incredible investment opportunity!", "BAD"),
]

print("Hedge Scoring Sanity Check")
print("=" * 60)
for text, label in test_cases:
    score = compute_hedge_score(text)
    flag = "✅" if (label == "GOOD" and score > 0.5) or (label == "BAD" and score < 0.5) else "❌"
    print(f"{flag} [{label}] Score={score:.3f}")
    print(f"   {text[:70]}..." if len(text) > 70 else f"   {text}")
    print()

---
## Stage 1: Supervised Fine-Tuning (SFT)

We fine-tune GPT-2 on financial text prompts so it learns the domain vocabulary 
before RL begins. Without SFT, PPO would be trying to shape incoherent outputs.

**Note**: For a real project, you'd use a financial corpus (e.g., SEC filings, 
analyst reports). Here we use a small synthetic dataset to keep things runnable.

In [ ]:
# ── SFT Training Data ─────────────────────────────────────────────────────────
# Mix of hedged + speculative text so the base model knows both styles.
# The reward model will later discriminate between them.

SFT_DATA = [
    # Hedged examples (preferred)
    "The company's revenue may decline if macroeconomic conditions worsen.",
    "Based on current data, earnings could potentially miss analyst estimates.",
    "The stock appears to be undervalued relative to historical price-to-earnings ratios.",
    "Market volatility suggests investors should consider diversifying their portfolios.",
    "Analysis indicates the sector might face headwinds given rising interest rates.",
    "The quarterly results could reflect ongoing supply chain disruptions.",
    "Depending on Federal Reserve policy, the bond market may experience fluctuations.",
    "Historical trends suggest that inflation tends to peak before a rate cycle ends.",
    "The merger appears contingent on regulatory approval from antitrust authorities.",
    "Subject to market conditions, the IPO could be delayed until Q2.",
    "Earnings growth may slow if consumer spending contracts in the coming quarters.",
    "The portfolio's risk exposure could increase if equity correlations rise sharply.",
    "Evidence suggests the company might need to restructure its debt obligations.",
    "The dividend yield appears sustainable, assuming cash flows remain stable.",
    "Current valuations may not fully reflect the risks of a potential recession.",
    
    # Speculative examples (present in SFT for diversity, penalized in RL)
    "This stock will definitely hit $500 by end of year!",
    "Guaranteed returns await investors who buy now before prices explode.",
    "The market is going to absolutely skyrocket next quarter.",
    "You cannot lose money on this incredible investment opportunity.",
    "Massive gains are certain for early investors in this sector.",
    
    # Neutral financial domain text
    "The Federal Reserve announced its decision on interest rates yesterday.",
    "Q3 earnings reports are expected from major banks next week.",
    "The S&P 500 closed lower amid concerns about inflation data.",
    "Analysts revised their price targets following the earnings release.",
    "Trading volume increased significantly during the market open.",
]

FINANCIAL_PROMPTS = [
    "The stock market",
    "Investors should consider",
    "Based on recent earnings",
    "The company's outlook",
    "Market analysis suggests",
    "The Federal Reserve",
    "Portfolio risk",
    "Quarterly results",
    "The sector may",
    "Bond yields",
    "Equity valuation",
    "The merger could",
    "Consumer spending",
    "Inflation data",
    "The IPO",
]

print(f"SFT dataset: {len(SFT_DATA)} samples")
print(f"RL prompts pool: {len(FINANCIAL_PROMPTS)} prompts")

In [ ]:
# ── Load Tokenizer ─────────────────────────────────────────────────────────────
MODEL_NAME = "gpt2"  # 124M params — runs on CPU in ~minutes

tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 has no pad token by default
tokenizer.padding_side = "left"  # For generation tasks

print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Model: {MODEL_NAME}")

In [ ]:
# ── Build SFT Dataset ─────────────────────────────────────────────────────────
def tokenize_sft(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        max_length=128,
        padding="max_length",
    )
    # For causal LM, labels = input_ids (predict next token)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

sft_dataset = Dataset.from_dict({"text": SFT_DATA})
sft_tokenized = sft_dataset.map(tokenize_sft, batched=True)
sft_tokenized.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print(f"SFT tokenized dataset: {len(sft_tokenized)} samples")

In [ ]:
# ── SFT Training ──────────────────────────────────────────────────────────────
sft_model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)
sft_model.config.pad_token_id = tokenizer.eos_token_id

sft_args = TrainingArguments(
    output_dir="./sft_gpt2_finance",
    num_train_epochs=5,           # Small dataset, more epochs
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    warmup_steps=10,
    logging_steps=10,
    save_strategy="no",
    report_to="none",             # Disable wandb/tensorboard
    fp16=torch.cuda.is_available(),
)

sft_trainer = Trainer(
    model=sft_model,
    args=sft_args,
    train_dataset=sft_tokenized,
)

print("Starting SFT training...")
sft_trainer.train()
print("SFT complete ✅")

In [ ]:
# ── Inspect SFT Output ────────────────────────────────────────────────────────
def generate_text(model, prompt: str, max_new_tokens: int = 50) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    model.eval()
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)

sft_model.to(device)
print("Sample outputs from SFT model (before RLHF):")
print("=" * 60)
for prompt in FINANCIAL_PROMPTS[:5]:
    text = generate_text(sft_model, prompt)
    score = compute_hedge_score(text)
    print(f"Prompt: '{prompt}'")
    print(f"Output: {text}")
    print(f"Hedge score: {score:.3f}")
    print()

---
## Stage 2: Reward Model Training

We train a **reward model** on synthetic pairwise preferences. This is the core 
of RLHF: instead of hand-crafting a reward function, we learn it from preference data.

### Pairwise preference setup
For each pair `(text_A, text_B)`, a human annotator says "I prefer A".  
We use our `compute_hedge_score` heuristic to simulate these annotations.

### Bradley-Terry loss
The reward model is trained with the **Bradley-Terry** preference model:

$$\mathcal{L}_{RM} = -\mathbb{E}_{(x, y_w, y_l)} \left[ \log \sigma(r(x, y_w) - r(x, y_l)) \right]$$

Where $y_w$ is the preferred (winner) and $y_l$ is the dispreferred (loser) completion.
This directly encodes: *the preferred completion should get a higher reward score.*

In [ ]:
# ── Generate Pairwise Preference Dataset ─────────────────────────────────────
# Strategy: for each financial prompt, generate two completions using the SFT 
# model, then label the more hedged one as "preferred" using our heuristic.
# In real RLHF, human annotators would do this labeling step.

# Supplementary pairs: hand-crafted for better coverage
PAIRWISE_DATA = [
    # (preferred/hedged, dispreferred/speculative)
    (
        "The stock may experience significant volatility if interest rates rise further.",
        "The stock will definitely skyrocket once interest rates change!"
    ),
    (
        "Based on historical data, earnings could potentially disappoint investors.",
        "Earnings will absolutely blow past expectations and guarantee huge gains."
    ),
    (
        "The merger appears contingent on regulatory approval and may face delays.",
        "The merger will certainly go through and profits are guaranteed to explode."
    ),
    (
        "Inflation data suggests the central bank might consider pausing rate hikes.",
        "The Fed will 100% cut rates causing an unstoppable rally in equities!"
    ),
    (
        "Consumer spending could weaken if unemployment rises in coming quarters.",
        "Consumer spending will definitely surge — buy now before massive gains!"
    ),
    (
        "The company's guidance seems cautious, possibly reflecting supply risks.",
        "Guidance will shock everyone! Incredible returns are absolutely guaranteed!"
    ),
    (
        "Portfolio risk may increase if correlations between asset classes rise.",
        "Your portfolio will skyrocket! This is a once-in-a-lifetime opportunity!"
    ),
    (
        "The sector appears to be facing headwinds from rising input costs.",
        "The sector will EXPLODE higher! Massive gains are coming for everyone!"
    ),
    (
        "Bond yields could rise further depending on inflation trajectory.",
        "Bond yields will definitely collapse! Don't miss out on amazing returns!"
    ),
    (
        "Analyst estimates suggest Q3 revenue might fall short of consensus.",
        "Revenue will certainly beat every estimate! Get rich quick on this trade!"
    ),
    (
        "The IPO valuation seems stretched relative to comparable companies.",
        "The IPO will make you rich! 100% gains are absolutely coming your way!"
    ),
    (
        "Dividend sustainability appears uncertain given declining free cash flow.",
        "Dividends will skyrocket! You can't lose money on this amazing stock!"
    ),
    (
        "The acquisition could create value if integration risks are managed well.",
        "This acquisition will definitely make billions! Guaranteed unbelievable returns!"
    ),
    (
        "Currency risk may dampen overseas earnings if the dollar strengthens.",
        "Currency moves will certainly boost earnings! Profits are guaranteed to soar!"
    ),
    (
        "The credit rating could be downgraded if debt levels continue to rise.",
        "Credit will absolutely be upgraded! Amazing gains await every investor!"
    ),
]

print(f"Pairwise preference pairs: {len(PAIRWISE_DATA)}")

# Verify the heuristic agrees with our hand labels
correct = 0
for preferred, dispreferred in PAIRWISE_DATA:
    s1 = compute_hedge_score(preferred)
    s2 = compute_hedge_score(dispreferred)
    if s1 > s2:
        correct += 1
        
print(f"Heuristic agrees with hand labels: {correct}/{len(PAIRWISE_DATA)} ({100*correct/len(PAIRWISE_DATA):.0f}%)")

In [ ]:
# ── Reward Model Architecture ─────────────────────────────────────────────────
# We use GPT2ForSequenceClassification with num_labels=1 (scalar reward output).
# The final linear layer projects from hidden_size → 1, giving us r(x).

reward_model = GPT2ForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1,  # Scalar reward
)
reward_model.config.pad_token_id = tokenizer.eos_token_id

# Initialize from SFT weights so reward model understands financial language
# Transfer all weights except the classification head
reward_model.transformer.load_state_dict(
    sft_model.transformer.state_dict()
)
reward_model = reward_model.to(device)
print("Reward model loaded with SFT backbone weights ✅")

In [ ]:
# ── Bradley-Terry Reward Model Trainer ───────────────────────────────────────
import torch.nn as nn
from torch.utils.data import DataLoader

class PreferenceDataset(torch.utils.data.Dataset):
    """Dataset of pairwise preferences for reward model training."""
    
    def __init__(self, pairs: List[Tuple[str, str]], tokenizer, max_length: int = 128):
        self.pairs = pairs
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.pairs)
    
    def __getitem__(self, idx):
        preferred, dispreferred = self.pairs[idx]
        
        # Tokenize both completions
        enc_w = self.tokenizer(
            preferred, truncation=True, max_length=self.max_length,
            padding="max_length", return_tensors="pt"
        )
        enc_l = self.tokenizer(
            dispreferred, truncation=True, max_length=self.max_length,
            padding="max_length", return_tensors="pt"
        )
        
        return {
            "input_ids_w": enc_w["input_ids"].squeeze(),
            "attention_mask_w": enc_w["attention_mask"].squeeze(),
            "input_ids_l": enc_l["input_ids"].squeeze(),
            "attention_mask_l": enc_l["attention_mask"].squeeze(),
        }


def bradley_terry_loss(reward_w: torch.Tensor, reward_l: torch.Tensor) -> torch.Tensor:
    """
    Bradley-Terry preference model loss.
    
    L = -log(sigmoid(r_w - r_l))
    
    Minimizing this pushes r_w > r_l, making the reward model
    assign higher scores to preferred (hedged) completions.
    """
    return -torch.log(torch.sigmoid(reward_w - reward_l)).mean()


def train_reward_model(
    model: nn.Module,
    pairs: List[Tuple[str, str]],
    tokenizer,
    num_epochs: int = 10,
    lr: float = 1e-5,
    batch_size: int = 4,
) -> List[float]:
    """Train reward model with Bradley-Terry loss."""
    
    dataset = PreferenceDataset(pairs, tokenizer)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    
    losses = []
    model.train()
    
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        for batch in loader:
            # Get rewards for preferred and dispreferred completions
            r_w = model(
                input_ids=batch["input_ids_w"].to(device),
                attention_mask=batch["attention_mask_w"].to(device)
            ).logits.squeeze(-1)  # shape: (batch,)
            
            r_l = model(
                input_ids=batch["input_ids_l"].to(device),
                attention_mask=batch["attention_mask_l"].to(device)
            ).logits.squeeze(-1)  # shape: (batch,)
            
            loss = bradley_terry_loss(r_w, r_l)
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            epoch_loss += loss.item()
        
        avg_loss = epoch_loss / len(loader)
        losses.append(avg_loss)
        
        if (epoch + 1) % 2 == 0:
            print(f"  Epoch {epoch+1:2d}/{num_epochs} | BT Loss: {avg_loss:.4f}")
    
    return losses


print("Training reward model with Bradley-Terry loss...")
rm_losses = train_reward_model(reward_model, PAIRWISE_DATA, tokenizer, num_epochs=10)
print("\nReward model training complete ✅")

In [ ]:
# ── Evaluate Reward Model ─────────────────────────────────────────────────────
def get_reward(model, text: str) -> float:
    """Get scalar reward from reward model for a given text."""
    model.eval()
    inputs = tokenizer(
        text, return_tensors="pt",
        truncation=True, max_length=128,
        padding="max_length"
    ).to(device)
    
    with torch.no_grad():
        reward = model(**inputs).logits.squeeze().item()
    return reward


print("Reward Model Evaluation")
print("=" * 65)
print(f"{'Text (truncated)':<45} {'RM Score':>8} {'Heuristic':>9}")
print("-" * 65)

eval_texts = [
    "The stock may decline if economic conditions worsen.",
    "Based on data, earnings could potentially miss estimates.",
    "This stock will definitely make you rich with guaranteed gains!",
    "BUY NOW! Skyrocket to the moon! 100% profit guaranteed!",
    "The company appears to face headwinds from rising costs.",
    "Massive unstoppable gains are absolutely certain this year!",
]

for text in eval_texts:
    rm_score = get_reward(reward_model, text)
    heuristic = compute_hedge_score(text)
    truncated = text[:43] + ".." if len(text) > 45 else text
    print(f"{truncated:<45} {rm_score:>8.3f} {heuristic:>9.3f}")

---
## Stage 3: RL Fine-Tuning with PPO — From Scratch

We implement **Proximal Policy Optimization (PPO)** entirely from scratch.  
No `trl.PPOTrainer`. Every component is hand-written and annotated.

### What we build

| Component | Description |
|-----------|-------------|
| `PolicyWithValueHead` | GPT-2 + a scalar value head `V(s)` |
| `RolloutBuffer` | Stores one full batch of rollout data |
| `compute_gae()` | GAE-λ advantage estimation |
| `compute_kl_penalty()` | Per-token KL from reference model |
| `ppo_epoch_update()` | One pass of the clipped PPO objective |
| `PPOTrainerScratch` | Orchestrates the full training loop |

### The math

**Clipped surrogate objective:**
$$L^{CLIP} = \mathbb{E}_t\left[\min\left(\rho_t \hat{A}_t,\ \text{clip}(\rho_t, 1-\varepsilon, 1+\varepsilon)\hat{A}_t\right)\right]$$

**Value function loss (Huber):**
$$L^{VF} = \mathbb{E}_t\left[\text{Huber}(V_\theta(s_t) - \hat{R}_t)\right]$$

**Entropy bonus** (prevents premature collapse):
$$L^{ENT} = \mathbb{E}_t\left[H(\pi_\theta(\cdot|s_t))\right]$$

**Total loss:**
$$L = -L^{CLIP} + c_1 L^{VF} - c_2 L^{ENT}$$

**KL-penalized reward (applied per-sequence):**
$$\tilde{r}(x, y) = r_{RM}(x, y) - \beta \sum_t \text{KL}_t(\pi_\theta \| \pi_{SFT})$$

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# COMPONENT 1: PolicyWithValueHead
# ══════════════════════════════════════════════════════════════════════════════
#
# PPO needs TWO outputs from the same model:
#   1. π(a|s)  — the language model head (logits over vocabulary)
#   2. V(s)    — a scalar value estimate (critic)
#
# We add a small MLP value head on top of GPT-2's last hidden states.
# The LM head (language model) is GPT-2's existing lm_head (unchanged).

class PolicyWithValueHead(nn.Module):
    """
    GPT-2 policy model augmented with a scalar value head for PPO.

    Architecture:
        GPT-2 backbone  →  hidden states  [B, T, H]
                                │
                    ┌───────────┴───────────┐
                    │                       │
              lm_head (frozen init)   value_head
              [H → vocab]             [H → 1]
                    │                       │
              logits [B,T,V]          values [B,T]
    """

    def __init__(self, gpt2_lm_model: GPT2LMHeadModel):
        super().__init__()
        # Reuse GPT-2's transformer blocks and LM head
        self.transformer = gpt2_lm_model.transformer  # GPT2Model
        self.lm_head = gpt2_lm_model.lm_head          # Linear(H → vocab)

        hidden_size = gpt2_lm_model.config.hidden_size  # 768 for gpt2

        # Value head: a small 2-layer MLP
        # Input: last hidden state at each token [H]
        # Output: scalar V(s_t) [1]
        self.value_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.Tanh(),
            nn.Linear(hidden_size // 2, 1),
        )

        # Weight init for value head: small values to avoid dominating early training
        for module in self.value_head:
            if isinstance(module, nn.Linear):
                nn.init.orthogonal_(module.weight, gain=0.01)
                nn.init.zeros_(module.bias)

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Returns:
            logits: [B, T, vocab_size]  — for computing log-probabilities
            values: [B, T]              — V(s) at each token position
        """
        outputs = self.transformer(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        hidden_states = outputs.last_hidden_state  # [B, T, H]

        logits = self.lm_head(hidden_states)                     # [B, T, V]
        values = self.value_head(hidden_states).squeeze(-1)      # [B, T]
        return logits, values

    @torch.no_grad()
    def generate(
        self,
        input_ids: torch.Tensor,
        max_new_tokens: int = 50,
        temperature: float = 0.8,
        top_p: float = 0.9,
        pad_token_id: int = 50256,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Greedy/nucleus sampling generation with log-prob tracking.

        Returns:
            full_ids:   [B, prompt_len + new_tokens]  — complete token sequence
            resp_ids:   [B, new_tokens]               — generated tokens only
            log_probs:  [B, new_tokens]               — log π(a_t | s_t)
        """
        B = input_ids.shape[0]
        cur = input_ids.clone()
        generated = []
        log_probs_list = []

        for _ in range(max_new_tokens):
            logits, _ = self.forward(cur)
            next_logits = logits[:, -1, :] / temperature    # [B, V]

            # Nucleus (top-p) sampling
            next_logits = _top_p_filter(next_logits, top_p)
            probs = F.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)  # [B, 1]

            log_prob = torch.log(probs.gather(1, next_token) + 1e-8)  # [B, 1]
            generated.append(next_token)
            log_probs_list.append(log_prob)

            cur = torch.cat([cur, next_token], dim=1)

            # Stop if all sequences produced EOS
            if (next_token == pad_token_id).all():
                break

        resp_ids = torch.cat(generated, dim=1)              # [B, T_new]
        log_probs = torch.cat(log_probs_list, dim=1)       # [B, T_new]
        return cur, resp_ids, log_probs


def _top_p_filter(logits: torch.Tensor, top_p: float) -> torch.Tensor:
    """Nucleus filtering: zero out tokens outside the top-p probability mass."""
    sorted_logits, sorted_idx = torch.sort(logits, descending=True)
    cumprobs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
    # Remove tokens with cumulative prob above threshold (shift right by 1 to keep first over)
    to_remove = (cumprobs - F.softmax(sorted_logits, dim=-1)) > top_p
    sorted_logits[to_remove] = float("-inf")
    # Scatter back to original order
    return sorted_logits.scatter(1, sorted_idx, sorted_logits)


print("PolicyWithValueHead defined ✅")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# COMPONENT 2: RolloutBuffer  +  compute_gae
# ══════════════════════════════════════════════════════════════════════════════
#
# The rollout buffer stores everything collected during one batch of generation.
# After filling it, we compute advantages via GAE before the gradient update.

@dataclass
class Rollout:
    """Stores data for a single (prompt, response) pair."""
    query_ids:   torch.Tensor    # [Q]        — prompt token ids
    response_ids: torch.Tensor   # [T]        — generated token ids
    log_probs_old: torch.Tensor  # [T]        — log π_old(a_t | s_t) at generation time
    values:      torch.Tensor    # [T]        — V_old(s_t) from value head
    reward:      float           # scalar     — final reward from reward model


class RolloutBuffer:
    """Accumulates rollouts for one PPO batch."""

    def __init__(self):
        self.rollouts: List[Rollout] = []

    def add(self, rollout: Rollout):
        self.rollouts.append(rollout)

    def clear(self):
        self.rollouts = []

    def __len__(self):
        return len(self.rollouts)


def compute_gae(
    rewards: List[float],
    values: torch.Tensor,
    gamma: float = 0.99,
    lam: float = 0.95,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Generalized Advantage Estimation (GAE-λ).

    For a sequence of length T with a single terminal reward at the last step:

        δ_t = r_t + γ V(s_{t+1}) - V(s_t)
        Â_t = δ_t + (γλ) δ_{t+1} + (γλ)² δ_{t+2} + ...

    In our LM setup, the reward is ONLY at the final token (sparse reward).
    Intermediate r_t = 0; r_T = rm_reward - kl_penalty.

    Args:
        rewards:  list of per-token rewards [T]  (mostly 0, last = rm_reward - kl)
        values:   V(s_t) for each token [T]      from value head
        gamma:    discount factor
        lam:      GAE smoothing parameter

    Returns:
        advantages: [T]
        returns:    [T]  = advantages + values  (target for value function)
    """
    T = len(rewards)
    advantages = torch.zeros(T, dtype=torch.float32)
    last_adv = 0.0

    # Bootstrap: V(s_{T+1}) = 0 (episode ends after generation)
    next_value = 0.0

    for t in reversed(range(T)):
        delta = rewards[t] + gamma * next_value - values[t].item()
        last_adv = delta + gamma * lam * last_adv
        advantages[t] = last_adv
        next_value = values[t].item()

    returns = advantages + values.detach()
    return advantages, returns


print("RolloutBuffer and compute_gae defined ✅")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# COMPONENT 3: KL Penalty  +  Reward Shaping
# ══════════════════════════════════════════════════════════════════════════════
#
# The KL penalty prevents the policy from drifting too far from the SFT reference.
# We compute it TOKEN-by-TOKEN and sum over the response sequence.
#
#   KL_t = Σ_v π_θ(v | s_t) * log[π_θ(v | s_t) / π_SFT(v | s_t)]
#
# This is the full KL, not just the selected-token approximation.
# We then sum KL_t across the response to get the sequence-level penalty.

def compute_kl_penalty(
    policy_logits: torch.Tensor,     # [T, vocab]  — from current policy
    ref_logits: torch.Tensor,        # [T, vocab]  — from frozen SFT reference
    reduction: str = "sum",          # "sum" or "mean" over tokens
) -> torch.Tensor:
    """
    Full KL divergence: KL(π_θ || π_ref) summed/averaged over T tokens.

    Uses the identity: KL(p||q) = Σ p * (log p - log q)
    """
    log_p = F.log_softmax(policy_logits, dim=-1)   # [T, V]
    log_q = F.log_softmax(ref_logits, dim=-1)      # [T, V]
    p = log_p.exp()

    # Per-token KL  [T]
    kl_per_token = (p * (log_p - log_q)).sum(dim=-1)   # [T]
    kl_per_token = kl_per_token.clamp(min=0)            # numerical safety

    if reduction == "sum":
        return kl_per_token.sum()
    elif reduction == "mean":
        return kl_per_token.mean()
    else:
        return kl_per_token


def shape_reward(
    rm_reward: float,
    policy_logits: torch.Tensor,   # [T, vocab]
    ref_logits: torch.Tensor,      # [T, vocab]
    kl_coef: float = 0.2,
) -> Tuple[float, float]:
    """
    Apply KL penalty to the scalar reward model score.

    Returns:
        shaped_reward: rm_reward - kl_coef * KL
        kl_value:      raw KL value (for logging)
    """
    with torch.no_grad():
        kl = compute_kl_penalty(policy_logits, ref_logits, reduction="sum")
        kl_value = kl.item()

    shaped = rm_reward - kl_coef * kl_value
    return shaped, kl_value


print("KL penalty and reward shaping defined ✅")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# COMPONENT 4: PPO Loss
# ══════════════════════════════════════════════════════════════════════════════
#
# The PPO clipped objective is the heart of the algorithm.
#
# Probability ratio:
#   ρ_t = π_θ(a_t | s_t) / π_old(a_t | s_t)
#       = exp(log π_θ - log π_old)
#
# Clipped surrogate:
#   L_CLIP = E[min(ρ_t * Â_t, clip(ρ_t, 1-ε, 1+ε) * Â_t)]
#
# We MAXIMIZE L_CLIP, so in PyTorch we MINIMIZE -L_CLIP.

def compute_log_probs_of_actions(
    logits: torch.Tensor,   # [T, vocab]
    actions: torch.Tensor,  # [T]  — the actual tokens generated
) -> torch.Tensor:
    """
    Compute log π(a_t | s_t) for a specific sequence of actions.
    
    This is how we re-evaluate the current policy's probability of
    the actions that were sampled during rollout.
    """
    log_probs = F.log_softmax(logits, dim=-1)          # [T, vocab]
    # Gather the log-prob of the chosen action at each step
    selected = log_probs.gather(1, actions.unsqueeze(1)).squeeze(1)  # [T]
    return selected


def ppo_loss(
    policy: "PolicyWithValueHead",
    rollouts: List[Rollout],
    clip_eps: float = 0.2,
    vf_coef: float = 0.1,
    ent_coef: float = 0.01,
    gamma: float = 0.99,
    lam: float = 0.95,
) -> Tuple[torch.Tensor, Dict[str, float]]:
    """
    Compute the PPO loss for a list of rollouts.

    Steps:
      1. For each rollout, re-run the CURRENT policy on the (query+response) sequence
      2. Extract log π_θ(a_t) for the response tokens
      3. Compute probability ratios ρ_t = exp(log π_θ - log π_old)
      4. Compute advantages via GAE
      5. Apply the clipped surrogate objective
      6. Add value function loss (Huber) and entropy bonus

    Returns:
        total_loss: scalar tensor (to call .backward() on)
        metrics:    dict with component losses and stats
    """
    total_policy_loss = 0.0
    total_value_loss  = 0.0
    total_entropy     = 0.0
    clip_frac         = 0.0
    n_tokens_total    = 0

    for rollout in rollouts:
        T = rollout.response_ids.shape[0]
        if T == 0:
            continue

        # Full sequence = [prompt tokens] + [response tokens]
        full_ids = torch.cat([
            rollout.query_ids,
            rollout.response_ids,
        ]).unsqueeze(0).to(device)  # [1, Q+T]

        # ── Re-evaluate current policy on this sequence ───────────────────
        logits_full, values_full = policy(full_ids)   # [1, Q+T, V], [1, Q+T]
        logits_full  = logits_full.squeeze(0)          # [Q+T, V]
        values_full  = values_full.squeeze(0)          # [Q+T]

        # We only care about the response tokens (positions Q-1 to Q+T-2)
        # because position t predicts token t+1.
        Q = rollout.query_ids.shape[0]
        resp_logits = logits_full[Q-1 : Q+T-1]        # [T, V]  shifted for next-token
        resp_values = values_full[Q-1 : Q+T-1]        # [T]

        # ── Log probs of the actions under current policy ─────────────────
        actions = rollout.response_ids.to(device)      # [T]
        log_probs_new = compute_log_probs_of_actions(resp_logits, actions)  # [T]

        # ── Probability ratio ρ_t ─────────────────────────────────────────
        log_probs_old = rollout.log_probs_old.to(device)  # [T]
        ratio = torch.exp(log_probs_new - log_probs_old)  # [T]

        # ── GAE advantage estimation ──────────────────────────────────────
        # Sparse reward: reward only at last token
        per_token_rewards = [0.0] * T
        per_token_rewards[-1] = rollout.reward        # final reward at last token

        advantages, returns = compute_gae(
            per_token_rewards,
            resp_values.detach(),
            gamma=gamma,
            lam=lam,
        )
        advantages = advantages.to(device)
        returns    = returns.to(device)

        # Normalize advantages within this rollout (reduces variance)
        if T > 1:
            advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        # ── Clipped surrogate loss ────────────────────────────────────────
        surr1 = ratio * advantages
        surr2 = ratio.clamp(1 - clip_eps, 1 + clip_eps) * advantages
        policy_loss = -torch.min(surr1, surr2).mean()   # negate: we maximize

        # ── Value function loss (Huber) ───────────────────────────────────
        value_loss = F.huber_loss(resp_values, returns.detach())

        # ── Entropy bonus ─────────────────────────────────────────────────
        # H(π) = -Σ π(a) log π(a)  — encourages exploration
        probs = F.softmax(resp_logits.detach(), dim=-1)
        log_probs_all = F.log_softmax(resp_logits.detach(), dim=-1)
        entropy = -(probs * log_probs_all).sum(dim=-1).mean()

        # ── Accumulate ────────────────────────────────────────────────────
        total_policy_loss += policy_loss
        total_value_loss  += value_loss
        total_entropy     += entropy
        clip_frac         += ((ratio - 1).abs() > clip_eps).float().mean().item()
        n_tokens_total    += T

    n = max(len(rollouts), 1)
    total_loss = (
        total_policy_loss / n
        + vf_coef  * (total_value_loss / n)
        - ent_coef * (total_entropy / n)
    )

    metrics = {
        "policy_loss": (total_policy_loss / n).item(),
        "value_loss":  (total_value_loss / n).item(),
        "entropy":     (total_entropy / n).item(),
        "clip_frac":   clip_frac / n,
    }
    return total_loss, metrics


print("ppo_loss defined ✅")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# COMPONENT 5: PPOTrainerScratch  — the full training loop
# ══════════════════════════════════════════════════════════════════════════════

class PPOTrainerScratch:
    """
    PPO trainer for RLHF — written from scratch.

    Algorithm (per step):
    ┌─────────────────────────────────────────────────────────────┐
    │ ROLLOUT PHASE (no grad)                                     │
    │   for each prompt in batch:                                 │
    │     1. Generate response with current policy π_θ            │
    │     2. Score response with reward model r_RM                │
    │     3. Compute KL penalty using reference model π_SFT       │
    │     4. shaped_reward = r_RM - β * KL                        │
    │     5. Compute V(s_t) from value head                       │
    │     6. Store (query, response, log_probs, values, reward)   │
    ├─────────────────────────────────────────────────────────────┤
    │ UPDATE PHASE (with grad)  — repeated ppo_epochs times       │
    │   for each mini-batch of rollouts:                          │
    │     1. Re-evaluate π_θ on stored sequences                  │
    │     2. Compute ρ_t = π_θ(a_t) / π_old(a_t)                 │
    │     3. Compute advantages via GAE                           │
    │     4. Compute clipped surrogate loss                       │
    │     5. Add value loss + entropy bonus                       │
    │     6. Gradient step                                        │
    └─────────────────────────────────────────────────────────────┘
    """

    def __init__(
        self,
        policy: PolicyWithValueHead,
        ref_model: GPT2LMHeadModel,         # frozen SFT reference
        reward_model: GPT2ForSequenceClassification,
        tokenizer: GPT2Tokenizer,
        prompts: List[str],
        # Hyperparameters
        lr: float = 1e-5,
        kl_coef: float = 0.2,
        clip_eps: float = 0.2,
        vf_coef: float = 0.1,
        ent_coef: float = 0.01,
        gamma: float = 0.99,
        lam: float = 0.95,
        ppo_epochs: int = 4,
        batch_size: int = 8,
        max_new_tokens: int = 50,
        temperature: float = 0.8,
        top_p: float = 0.9,
    ):
        self.policy       = policy.to(device)
        self.ref_model    = ref_model.to(device)
        self.reward_model = reward_model.to(device)
        self.tokenizer    = tokenizer
        self.prompts      = prompts

        self.kl_coef      = kl_coef
        self.clip_eps     = clip_eps
        self.vf_coef      = vf_coef
        self.ent_coef     = ent_coef
        self.gamma        = gamma
        self.lam          = lam
        self.ppo_epochs   = ppo_epochs
        self.batch_size   = batch_size
        self.max_new_tokens = max_new_tokens
        self.temperature  = temperature
        self.top_p        = top_p

        self.optimizer    = torch.optim.AdamW(policy.parameters(), lr=lr)
        self.buffer       = RolloutBuffer()

        # Freeze reference model
        for p in self.ref_model.parameters():
            p.requires_grad = False

    # ── Rollout Phase ────────────────────────────────────────────────────────

    @torch.no_grad()
    def _collect_rollouts(self, prompt_batch: List[str]) -> List[Dict]:
        """
        Generate one response per prompt and score it.
        Returns list of raw info dicts (before GAE).
        """
        self.policy.eval()
        rollout_data = []

        for prompt in prompt_batch:
            # Tokenize prompt
            enc = self.tokenizer(
                prompt, return_tensors="pt",
                truncation=True, max_length=32,
            ).to(device)
            query_ids = enc["input_ids"].squeeze(0)   # [Q]

            # Generate with current policy
            full_ids, resp_ids, log_probs_old = self.policy.generate(
                query_ids.unsqueeze(0),
                max_new_tokens=self.max_new_tokens,
                temperature=self.temperature,
                top_p=self.top_p,
                pad_token_id=self.tokenizer.eos_token_id,
            )
            resp_ids      = resp_ids.squeeze(0)       # [T]
            log_probs_old = log_probs_old.squeeze(0)  # [T]

            # Get value estimates from policy's value head
            logits_full, values_full = self.policy(full_ids)
            Q = query_ids.shape[0]
            T = resp_ids.shape[0]
            values_resp = values_full.squeeze(0)[Q-1 : Q+T-1]   # [T]

            # Get reference model logits for KL computation
            ref_logits_full, _ = self.ref_model(full_ids)
            ref_logits_resp = ref_logits_full.squeeze(0)[Q-1 : Q+T-1]  # [T, V]
            policy_logits_resp = logits_full.squeeze(0)[Q-1 : Q+T-1]   # [T, V]

            # Score with reward model
            full_text = self.tokenizer.decode(resp_ids, skip_special_tokens=True)
            rm_reward = get_reward(self.reward_model, full_text)

            # Shape reward: subtract KL penalty
            shaped_reward, kl_val = shape_reward(
                rm_reward, policy_logits_resp, ref_logits_resp, self.kl_coef
            )

            rollout_data.append({
                "query_ids":    query_ids.cpu(),
                "response_ids": resp_ids.cpu(),
                "log_probs_old": log_probs_old.cpu(),
                "values":       values_resp.cpu(),
                "rm_reward":    rm_reward,
                "kl":           kl_val,
                "reward":       shaped_reward,
                "text":         full_text,
            })

        return rollout_data

    # ── Update Phase ─────────────────────────────────────────────────────────

    def _ppo_update(self, rollouts: List[Rollout]) -> Dict[str, float]:
        """Run ppo_epochs gradient steps on the collected rollouts."""
        self.policy.train()
        all_metrics = []

        for epoch in range(self.ppo_epochs):
            # Shuffle rollouts each inner epoch
            random.shuffle(rollouts)
            loss, metrics = ppo_loss(
                self.policy,
                rollouts,
                clip_eps=self.clip_eps,
                vf_coef=self.vf_coef,
                ent_coef=self.ent_coef,
                gamma=self.gamma,
                lam=self.lam,
            )
            self.optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.policy.parameters(), 1.0)
            self.optimizer.step()
            all_metrics.append(metrics)

        # Average metrics over inner epochs
        avg = {k: np.mean([m[k] for m in all_metrics]) for k in all_metrics[0]}
        return avg

    # ── Main Training Loop ────────────────────────────────────────────────────

    def train(self, num_steps: int = 30) -> List[Dict]:
        """
        Run the RLHF training loop for num_steps outer PPO steps.

        Each step:
          1. Sample a batch of prompts
          2. Collect rollouts (generate + score)
          3. Run ppo_epochs gradient updates
          4. Log and return metrics
        """
        training_log = []
        prompt_idx = 0

        print(f"Starting PPO-from-scratch training for {num_steps} steps...")
        print("=" * 70)

        for step in range(num_steps):
            # ── 1. Sample prompts ────────────────────────────────────────
            batch_prompts = []
            for _ in range(self.batch_size):
                batch_prompts.append(self.prompts[prompt_idx % len(self.prompts)])
                prompt_idx += 1

            # ── 2. Collect rollouts ──────────────────────────────────────
            rollout_data = self._collect_rollouts(batch_prompts)

            rollouts = [
                Rollout(
                    query_ids=rd["query_ids"],
                    response_ids=rd["response_ids"],
                    log_probs_old=rd["log_probs_old"],
                    values=rd["values"],
                    reward=rd["reward"],
                )
                for rd in rollout_data
            ]

            # ── 3. PPO gradient update ───────────────────────────────────
            update_metrics = self._ppo_update(rollouts)

            # ── 4. Logging ───────────────────────────────────────────────
            mean_rm_reward  = np.mean([rd["rm_reward"] for rd in rollout_data])
            mean_kl         = np.mean([rd["kl"] for rd in rollout_data])
            mean_shaped_rew = np.mean([rd["reward"] for rd in rollout_data])
            mean_hedge      = np.mean([compute_hedge_score(rd["text"]) for rd in rollout_data])

            log_entry = {
                "step":           step,
                "mean_rm_reward": mean_rm_reward,
                "mean_kl":        mean_kl,
                "mean_shaped_reward": mean_shaped_rew,
                "hedge_score":    mean_hedge,
                **update_metrics,
            }
            training_log.append(log_entry)

            if (step + 1) % 5 == 0:
                print(
                    f"Step {step+1:3d} | "
                    f"RM Reward: {mean_rm_reward:+.3f} | "
                    f"KL: {mean_kl:.3f} | "
                    f"Shaped: {mean_shaped_rew:+.3f} | "
                    f"Hedge: {mean_hedge:.3f} | "
                    f"ClipFrac: {update_metrics['clip_frac']:.3f}"
                )

        print("\nPPO-from-scratch training complete ✅")
        return training_log


# ── Instantiate models for PPO-from-scratch ───────────────────────────────────
# Policy: GPT2 + value head, initialized from SFT weights
policy_for_ppo = PolicyWithValueHead(sft_model).to(device)

# Reference: same SFT model, frozen (used only for KL computation)
ref_gpt2 = GPT2LMHeadModel.from_pretrained(MODEL_NAME)
ref_gpt2.transformer.load_state_dict(sft_model.transformer.state_dict())
ref_gpt2.lm_head.load_state_dict(sft_model.lm_head.state_dict())
ref_gpt2 = ref_gpt2.to(device)
for p in ref_gpt2.parameters():
    p.requires_grad = False

# The reference model also needs a value head for our PolicyWithValueHead forward call
# We wrap it too (value head weights don't matter since we only use logits from ref)
ref_policy = PolicyWithValueHead(ref_gpt2).to(device)
for p in ref_policy.parameters():
    p.requires_grad = False

print("Policy (from-scratch PPO):  ✅")
print("Reference model (frozen):   ✅")


# ── Create and run the PPO trainer ────────────────────────────────────────────
ppo_scratch = PPOTrainerScratch(
    policy       = policy_for_ppo,
    ref_model    = ref_policy,
    reward_model = reward_model,
    tokenizer    = tokenizer,
    prompts      = EXPANDED_PROMPTS,
    lr           = 1e-5,
    kl_coef      = 0.2,
    clip_eps     = 0.2,
    vf_coef      = 0.1,
    ent_coef     = 0.01,
    gamma        = 0.99,
    lam          = 0.95,
    ppo_epochs   = 4,
    batch_size   = 8,
    max_new_tokens = 50,
)

training_log = ppo_scratch.train(num_steps=30)

---
## Evaluation: Before vs After RLHF

In [ ]:
# ── Side-by-side comparison: SFT vs PPO-from-scratch ─────────────────────────
import textwrap

def generate_rlhf_scratch(model: PolicyWithValueHead, prompt: str, max_new_tokens: int = 60) -> str:
    """Generate text from the PPO-from-scratch trained policy model."""
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    query_ids = enc["input_ids"]   # [1, Q]

    model.eval()
    full_ids, resp_ids, _ = model.generate(
        query_ids,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )
    # Decode full sequence (prompt + response)
    return tokenizer.decode(full_ids.squeeze(0), skip_special_tokens=True)


eval_prompts = [
    "The stock market",
    "Investors should consider",
    "The company's outlook",
    "Market analysis suggests",
    "The Federal Reserve",
]

print("RLHF EVALUATION: SFT vs PPO-from-scratch Policy")
print("=" * 70)

results = []
for prompt in eval_prompts:
    sft_text  = generate_text(sft_model, prompt)
    rlhf_text = generate_rlhf_scratch(policy_for_ppo, prompt)

    sft_score  = compute_hedge_score(sft_text)
    rlhf_score = compute_hedge_score(rlhf_text)
    improvement = rlhf_score - sft_score

    results.append({
        "prompt":      prompt,
        "sft_score":   sft_score,
        "rlhf_score":  rlhf_score,
        "improvement": improvement,
    })

    print(f"\nPrompt: '{prompt}'")
    print(f"  SFT  [{sft_score:.3f}]: {textwrap.shorten(sft_text, 80)}")
    print(f"  RLHF [{rlhf_score:.3f}]: {textwrap.shorten(rlhf_text, 80)}")
    delta_str = f"+{improvement:.3f}" if improvement > 0 else f"{improvement:.3f}"
    flag = "✅" if improvement > 0 else "⚠️"
    print(f"  Change: {delta_str} {flag}")

print("\n" + "=" * 70)
avg_improvement = np.mean([r["improvement"] for r in results])
print(f"Average hedge score improvement: {avg_improvement:+.3f}")

In [ ]:
# ── Plot Training Curves ──────────────────────────────────────────────────────
import matplotlib.pyplot as plt

if training_log:
    steps       = [d["step"] for d in training_log]
    rm_rewards  = [d["mean_rm_reward"] for d in training_log]
    kls         = [d["mean_kl"] for d in training_log]
    hedges      = [d["hedge_score"] for d in training_log]
    clip_fracs  = [d["clip_frac"] for d in training_log]
    pol_losses  = [d["policy_loss"] for d in training_log]
    val_losses  = [d["value_loss"] for d in training_log]

    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    fig.suptitle(
        "RLHF PPO From Scratch: Aligning GPT-2 for Hedged Financial Language",
        fontsize=13, fontweight="bold"
    )

    def _plot(ax, x, y, color, title, ylabel, hline=None):
        ax.plot(x, y, color=color, linewidth=2)
        ax.set_title(title)
        ax.set_xlabel("PPO Step")
        ax.set_ylabel(ylabel)
        ax.grid(True, alpha=0.3)
        if hline is not None:
            ax.axhline(y=hline, color="gray", linestyle="--", alpha=0.5)

    _plot(axes[0,0], steps, rm_rewards,  "#2196F3", "RM Reward",            "Reward",       hline=0)
    _plot(axes[0,1], steps, kls,         "#FF5722", "KL Divergence (β·KL)", "KL")
    _plot(axes[0,2], steps, hedges,      "#4CAF50", "Hedge Score (heuristic)", "Score [0,1]")
    axes[0,2].set_ylim(0, 1)
    _plot(axes[1,0], steps, clip_fracs,  "#9C27B0", "Clip Fraction",        "Fraction")
    _plot(axes[1,1], steps, pol_losses,  "#F44336", "Policy Loss",          "Loss")
    _plot(axes[1,2], steps, val_losses,  "#FF9800", "Value Loss",           "Loss")

    plt.tight_layout()
    plt.savefig("rlhf_ppo_scratch_curves.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("Training curves saved to rlhf_ppo_scratch_curves.png ✅")

---
## Understanding Reward Hacking

A critical concept in RLHF. Watch what happens when we run PPO **without** the KL penalty.

In [ ]:
# ── Reward Hacking Demo ───────────────────────────────────────────────────────
# Demonstrates WHY the KL penalty is essential.
# Without it, gradient ascent on the reward heuristic finds degenerate solutions.

reward_hacked_examples = [
    # Stuffing hedge words — games our heuristic without coherent meaning
    "May might could possibly potentially uncertain risk volatility if depending on contingent subject to change.",
    # Hedge words but completely incoherent
    "The stock may could possibly might uncertain possibly may could depending on if.",
    # Genuine hedged text (what we actually want)
    "The stock may decline if inflation persists beyond the current quarter.",
    # Coherent but no hedging
    "The stock rose sharply after earnings were released yesterday.",
]

print("Reward Hacking Illustration")
print("=" * 65)
print("Shows why KL penalty matters — the heuristic reward can be gamed")
print()
print(f"{'Type':<20} {'Heuristic':>9} {'Text (truncated)'}")
print("-" * 65)
labels = ["HACKED (gamed)", "HACKED (gamed)", "GENUINE hedge", "No hedge"]
for text, label in zip(reward_hacked_examples, labels):
    score = compute_hedge_score(text)
    truncated = text[:35] + "..." if len(text) > 35 else text
    print(f"{label:<20} {score:>9.3f} {truncated}")

KL_COEF = ppo_scratch.kl_coef
print()
print("⚠️  The KL penalty prevents the model from drifting toward")
print("    incoherent hedge-word stuffing by anchoring it to SFT.")
print()
print("   R_total = r_RM(text) - β × KL(π_θ || π_SFT)")
print(f"   With β = {KL_COEF}, incoherent text gets a heavy KL penalty.")
print()
print("   How our from-scratch implementation enforces this:")
print("   • KL computed token-by-token: KL_t = Σ_v π_θ(v|s_t) log[π_θ/π_SFT]")
print("   • Summed over entire response and subtracted from r_RM before GAE")
print("   • Even if hacked text has high r_RM, its KL term dominates")

---
## Summary & Key Takeaways

### What we built (all from scratch)
| Stage | Component | Purpose |
|-------|-----------|----------|
| SFT | GPT-2 fine-tuned on financial text | Teaches domain vocabulary |
| RM | GPT-2 classification head + Bradley-Terry loss | Encodes human preference for hedged language |
| PPO | `PolicyWithValueHead` + `RolloutBuffer` + `compute_gae` + `ppo_loss` + `PPOTrainerScratch` | Full RL optimization, no trl dependency |

### PPO from-scratch components explained

**1. `PolicyWithValueHead`** — wraps GPT-2 with a 2-layer MLP value head. The LM head outputs `π(a|s)` (action probabilities); the value head outputs `V(s)` (baseline for advantage estimation).

**2. `RolloutBuffer` + `Rollout`** — store `(query, response, log_probs_old, values, reward)` collected during the generation phase. `log_probs_old` are crucial: they capture `π_θ_old` so we can compute the probability ratio `ρ_t = π_θ / π_θ_old` without storing the full old model.

**3. `compute_gae()`** — Generalized Advantage Estimation (GAE-λ). Given sparse terminal rewards, propagates the reward signal backwards through time with exponential discounting. Controls variance/bias via λ.

**4. `compute_kl_penalty()`** — full KL divergence `KL(π_θ || π_SFT)` computed per-token over the full vocabulary, then summed over the response. Prevents reward hacking by penalizing drift from the SFT reference.

**5. `ppo_loss()`** — implements all three loss terms:
   - **Clipped surrogate**: `min(ρ·Â, clip(ρ, 1-ε, 1+ε)·Â)` — the core PPO constraint
   - **Value loss** (Huber): trains the critic to predict returns accurately  
   - **Entropy bonus**: discourages premature policy collapse

**6. `PPOTrainerScratch`** — orchestrates the full loop: rollout → KL-penalized reward → GAE advantages → `ppo_epochs` gradient updates per batch.

### Extensions to try
- Add **batch-level advantage normalization** (current: per-rollout)  
- Implement **adaptive KL control** (auto-adjust β to keep KL near a target)  
- Try **multiple mini-batches** per rollout batch for sample efficiency  
- Replace heuristic reward with a trained classifier on real financial corpora  
- Experiment with different γ/λ values in GAE and observe stability effects  
- Try **DPO** (Direct Preference Optimization) as an RL-free alternative